# Chapter 3 — Decorator Pattern
## *Decorating Objects*

**Intent:** Attach additional responsibilities to an object **dynamically**.  
Decorators provide a flexible alternative to subclassing for extending functionality.

### OO Design Principle
- **Open/Closed Principle:** Classes should be open for extension, closed for modification.

### Book context
Starbuzz Coffee: `Beverage` base class. Adding condiments (milk, mocha, whip, soy) via subclassing leads to a class explosion.  
Solution: wrap beverages in condiment decorators — each adds its cost and description.

## Python building block: `@property`

`@property` lets you access a method **as if it were an attribute** — no parentheses needed at the call site.

```python
# Without @property — must call it like a function
drink.description()   # "Espresso"

# With @property — accessed like a plain attribute
drink.description     # "Espresso"
```

### Why use it?

It keeps the public API clean. Callers don't need to know or care whether `description` is stored data or computed on the fly — it just looks like an attribute either way.

```python
class Espresso(Beverage):
    @property
    def description(self):
        return "Espresso"

e = Espresso()
print(e.description)   # "Espresso" — no () needed
```

### `@property` + `@abstractmethod` together

In this notebook you'll see both stacked:

```python
@property
@abstractmethod
def description(self) -> str: ...
```

This means: *"subclasses must implement `description`, and it must be exposed as a property (not a regular method call)."* The order matters — `@property` always goes on top.

In [ ]:
from abc import ABC, abstractmethod

# ── Component interface ──────────────────────────────────────────────────────
class Beverage(ABC):
    @property
    @abstractmethod
    def description(self) -> str: ...

    @abstractmethod
    def cost(self) -> float: ...


# ── Concrete components ──────────────────────────────────────────────────────
class Espresso(Beverage):
    @property
    def description(self): return "Espresso"
    def cost(self): return 1.99

class HouseBlend(Beverage):
    @property
    def description(self): return "House Blend Coffee"
    def cost(self): return 0.89

class DarkRoast(Beverage):
    @property
    def description(self): return "Dark Roast"
    def cost(self): return 0.99

In [ ]:
# ── Decorator base ───────────────────────────────────────────────────────────
class CondimentDecorator(Beverage, ABC):
    def __init__(self, beverage: Beverage):
        self._beverage = beverage  # wraps the component


# ── Concrete decorators ──────────────────────────────────────────────────────
class Mocha(CondimentDecorator):
    @property
    def description(self):
        return self._beverage.description + ", Mocha"
    def cost(self): return self._beverage.cost() + 0.20

class Whip(CondimentDecorator):
    @property
    def description(self):
        return self._beverage.description + ", Whip"
    def cost(self): return self._beverage.cost() + 0.10

class Soy(CondimentDecorator):
    @property
    def description(self):
        return self._beverage.description + ", Soy"
    def cost(self): return self._beverage.cost() + 0.15

class SteamedMilk(CondimentDecorator):
    @property
    def description(self):
        return self._beverage.description + ", Steamed Milk"
    def cost(self): return self._beverage.cost() + 0.10

In [ ]:
# ── Demo ─────────────────────────────────────────────────────────────────────
def print_order(b: Beverage):
    print(f"{b.description:40s} ${b.cost():.2f}")

# Plain espresso
print_order(Espresso())

# Dark roast + double mocha + whip
drink = DarkRoast()
drink = Mocha(drink)
drink = Mocha(drink)   # double mocha
drink = Whip(drink)
print_order(drink)

# House blend + soy + mocha + whip
drink2 = HouseBlend()
drink2 = Soy(drink2)
drink2 = Mocha(drink2)
drink2 = Whip(drink2)
print_order(drink2)

## Python's `@functools.wraps` and `@property` ARE decorators

Python's `@decorator` syntax wraps functions — same structural idea, applied to functions.

In [ ]:
import functools, time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

def logger(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with {args}, {kwargs}")
        return func(*args, **kwargs)
    return wrapper

# Stacking decorators = wrapping wrappers, just like condiments
@timer
@logger
def compute(n: int) -> int:
    return sum(range(n))

compute(1_000_000)

## stdlib example: `io` module
`BufferedReader(FileIO(...))` — Java's I/O classes and Python's `io` hierarchy both use Decorator.

In [ ]:
import io

# FileIO → BufferedReader → TextIOWrapper: three layers of decoration
with open("/dev/null", "rb") as raw:
    buffered = io.BufferedReader(raw)
    print(type(buffered))   # BufferedReader wraps RawIOBase

## Interview cheat-sheet

| Question | Answer |
|---|---|
| Problem it solves? | Class explosion from subclassing every combination of features. |
| Key structural requirement? | Decorator **must** share the same interface/supertype as what it wraps. |
| Downside? | Can create many small objects; order of decoration matters; hard to inspect the chain. |
| vs. Inheritance? | Decorator adds behavior at runtime to specific instances; inheritance is compile-time and affects all instances. |

**Pattern family:** Structural